In [1]:
#1 Installing and Loading Required R Packages
install.packages("sqldf")
install.packages("dplyr")
install.packages("ggplot2")

library(sqldf)
library(dplyr)
library(ggplot2)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
#2 Loading Cleaned Datasets
customers <- read.csv("cleaned_customers.csv")
orders <- read.csv("cleaned_orders.csv")
deliveries <- read.csv("cleaned_deliveries.csv")
drivers <- read.csv("cleaned_drivers.csv")
vehicles <- read.csv("cleaned_vehicles.csv")
hubs <- read.csv("cleaned_hubs.csv")
complaints <- read.csv("cleaned_complaints.csv")
incidents <- read.csv("cleaned_incidents.csv")
app_events <- read.csv("cleaned_app_events.csv")

print("All cleaned datasets loaded successfully")

[1] "All cleaned datasets loaded successfully"


In [3]:
#3 Checking Dataset Structure
str(customers)
str(orders)
str(deliveries)

'data.frame':	650 obs. of  9 variables:
 $ customer_id         : chr  "C0001" "C0002" "C0003" "C0004" ...
 $ age                 : int  26 61 66 75 26 41 54 70 34 23 ...
 $ home_zone           : chr  "North" "AIRPORT" "East" "CENTRAL" ...
 $ customer_type       : chr  "SME" "Consumer" "Consumer" "Consumer" ...
 $ signup_date         : chr  "2024-11-27 04:25:00" "2025-10-28 01:04:00" "2025-07-02 03:23:00" "2025-08-19 01:58:00" ...
 $ loyalty_score       : num  44.9 55.4 75.9 32.5 55.9 39.9 36.1 84.6 62.6 87.2 ...
 $ app_engagement_score: num  69.2 66.6 33.8 33 100 43.3 39 65.2 40.8 48.6 ...
 $ preferred_channel   : chr  "App" "App" "" "App" ...
 $ account_status      : chr  "Active" "Active" "Active" "Active" ...
'data.frame':	1250 obs. of  12 variables:
 $ order_id             : chr  "O00001" "O00002" "O00003" "O00004" ...
 $ customer_id          : chr  "C0292" "C0459" "C0161" "C0520" ...
 $ service_type         : chr  "Passenger" "Passenger" "Passenger" "Parcel" ...
 $ order_created_a

In [4]:
#4 SQL Query 1: Overall Delivery Status
query1 <- sqldf("
SELECT
    delivery_status,
    COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY delivery_status
ORDER BY total_deliveries DESC
")

query1

delivery_status,total_deliveries
<chr>,<int>
OnTime,616
Delayed,202
Failed,132


In [5]:
#5 SQL Query 2: Average Delivery Duration by Status
query2 <- sqldf("
SELECT
    delivery_status,
    ROUND(AVG(delivery_duration_hours), 2) AS avg_delivery_duration_hours
FROM deliveries
GROUP BY delivery_status
ORDER BY avg_delivery_duration_hours DESC
")

query2

delivery_status,avg_delivery_duration_hours
<chr>,<dbl>
Failed,17.78
Delayed,13.51
OnTime,6.49


In [6]:
#6 SQL Query 3: Hub Performance Analysis
query3 <- sqldf("
SELECT
    h.hub_name,
    h.zone,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries
FROM deliveries d
JOIN hubs h ON d.hub_id = h.hub_id
GROUP BY h.hub_name, h.zone
ORDER BY failed_deliveries DESC
")

query3

hub_name,zone,total_deliveries,delayed_deliveries,failed_deliveries
<chr>,<chr>,<int>,<int>,<int>
Midtown Relay,Central,128,22,26
Central Core,Central,115,25,23
North Exchange,North,136,26,17
West Gate,West,127,28,16
Airport Hub,Airport,104,27,15
Riverside Hub,Riverside,115,25,14
East Dock,East,119,23,11
South Link,South,106,26,10


In [9]:
#7 SQL Query 4: Driver Performance Analysis

query4 <- sqldf("
SELECT
    dr.driver_id,
    dr.years_experience,
    dr.driver_rating,
    dr.training_score,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries
FROM deliveries d
JOIN drivers dr ON d.driver_id = dr.driver_id
GROUP BY dr.driver_id, dr.years_experience, dr.driver_rating, dr.training_score
ORDER BY failed_deliveries DESC
LIMIT 10
")

query4

driver_id,years_experience,driver_rating,training_score,total_deliveries,delayed_deliveries,failed_deliveries
<chr>,<int>,<dbl>,<dbl>,<int>,<int>,<int>
D024,8,3.35,71.4,8,0,4
D104,15,3.45,87.7,7,0,4
D133,12,3.99,88.2,12,2,4
D004,13,4.75,88.9,9,1,3
D010,8,3.95,70.0,7,0,3
D055,15,5.00,90.5,10,2,3
D083,12,4.16,80.8,9,1,3
D092,15,4.24,88.2,5,0,3
D108,10,4.33,70.6,11,0,3


In [10]:
#8 SQL Query 5: Vehicle Maintenance and Delivery Failure
query5 <- sqldf("
SELECT
    v.maintenance_status,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    ROUND((SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0) / COUNT(d.delivery_id), 2) AS failure_rate
FROM deliveries d
JOIN vehicles v ON d.vehicle_id = v.vehicle_id
GROUP BY v.maintenance_status
ORDER BY failure_rate DESC
")

query5

maintenance_status,total_deliveries,failed_deliveries,failure_rate
<chr>,<int>,<int>,<dbl>
InRepair,254,77,30.31
Active,542,45,8.30
Scheduled,154,10,6.49


In [11]:
#9 SQL Query 6: Complaint Type Analysis
query6 <- sqldf("
SELECT
    complaint_type,
    COUNT(*) AS total_complaints,
    ROUND(AVG(compensation_amount), 2) AS avg_compensation
FROM complaints
GROUP BY complaint_type
ORDER BY total_complaints DESC
")

query6

complaint_type,total_complaints,avg_compensation
<chr>,<int>,<dbl>
Delay,101,18.05
MissedPickup,64,22.59
AppIssue,53,19.61
DriverBehaviour,51,21.15
SupportExperience,20,17.13
Billing,16,23.87
Damage,15,23.98


In [12]:
#10 SQL Query 7: Delivery Status and Complaint Analysis
query7 <- sqldf("
SELECT
    d.delivery_status,
    COUNT(DISTINCT d.delivery_id) AS total_deliveries,
    COUNT(c.complaint_id) AS total_complaints,
    ROUND(AVG(c.compensation_amount), 2) AS avg_compensation
FROM deliveries d
JOIN orders o ON d.order_id = o.order_id
LEFT JOIN complaints c ON o.order_id = c.order_id
GROUP BY d.delivery_status
ORDER BY total_complaints DESC
")

query7

delivery_status,total_deliveries,total_complaints,avg_compensation
<chr>,<int>,<int>,<dbl>
OnTime,616,149,19.18
Delayed,202,48,18.36
Failed,132,35,25.47


In [13]:
#11 SQL Query 8: Incident Severity Analysis
query8 <- sqldf("
SELECT
    i.severity,
    COUNT(i.incident_id) AS total_incidents,
    COUNT(d.delivery_id) AS linked_deliveries
FROM incidents i
LEFT JOIN deliveries d ON i.delivery_id = d.delivery_id
GROUP BY i.severity
ORDER BY total_incidents DESC
")

query8

severity,total_incidents,linked_deliveries
<chr>,<int>,<int>
Medium,106,106
Low,79,79
High,68,68
Critical,27,27


In [14]:
#12 SQL Query 9: App Event Failure Rate
query9 <- sqldf("
SELECT
    event_type,
    COUNT(*) AS total_events,
    SUM(CASE WHEN success_flag = 0 THEN 1 ELSE 0 END) AS failed_events,
    ROUND((SUM(CASE WHEN success_flag = 0 THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS failure_rate
FROM app_events
GROUP BY event_type
ORDER BY failure_rate DESC
")

query9

event_type,total_events,failed_events,failure_rate
<chr>,<int>,<int>,<dbl>
chat_escalated,38,19,50.00
payment_retry,69,19,27.54
track_order,138,0,0.00
search_route,99,0,0.00
eta_refresh,105,0,0.00
delivery_instruction_update,75,0,0.00
chat_opened,88,0,0.00
cancel_attempt,28,0,0.00


In [15]:
#13 SQL Query 10: Hub Volume, Complaints and Failures
query10 <- sqldf("
SELECT
    h.hub_name,
    h.zone,
    COUNT(DISTINCT d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    COUNT(c.complaint_id) AS total_complaints
FROM hubs h
JOIN deliveries d ON h.hub_id = d.hub_id
JOIN orders o ON d.order_id = o.order_id
LEFT JOIN complaints c ON o.order_id = c.order_id
GROUP BY h.hub_name, h.zone
ORDER BY total_complaints DESC
")

query10

hub_name,zone,total_deliveries,failed_deliveries,total_complaints
<chr>,<chr>,<int>,<int>,<int>
Midtown Relay,Central,128,27,35
East Dock,East,119,11,33
Riverside Hub,Riverside,115,14,33
North Exchange,North,136,18,32
Central Core,Central,115,23,30
West Gate,West,127,16,28
Airport Hub,Airport,104,15,23
South Link,South,106,10,18


In [16]:
#14 Saving SQL Query Outputs
write.csv(query1, "sql_query1_delivery_status.csv", row.names = FALSE)
write.csv(query2, "sql_query2_delivery_duration.csv", row.names = FALSE)
write.csv(query3, "sql_query3_hub_performance.csv", row.names = FALSE)
write.csv(query4, "sql_query4_driver_performance.csv", row.names = FALSE)
write.csv(query5, "sql_query5_vehicle_failure.csv", row.names = FALSE)
write.csv(query6, "sql_query6_complaints.csv", row.names = FALSE)
write.csv(query7, "sql_query7_delivery_complaints.csv", row.names = FALSE)
write.csv(query8, "sql_query8_incidents.csv", row.names = FALSE)
write.csv(query9, "sql_query9_app_events.csv", row.names = FALSE)
write.csv(query10, "sql_query10_hub_complaints.csv", row.names = FALSE)

print("All SQL query outputs saved successfully")

[1] "All SQL query outputs saved successfully"
